# Modul B · Kapitel 1.3 — Chain-of-Thought

## Challenge: Zählaufgaben in Security-Logs zuverlässig lösen


**Lernziel:** Du baust Zero-Shot-CoT und Few-Shot-CoT für dieselbe Zählaufgabe und vergleichst beide auf vorbereiteten Testdaten.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

Es gibt genau **zwei Challenges**: einen Zero-Shot-CoT-Prompt und einen Few-Shot-CoT-Prompt.


---
## 0 · Setup

▶️ Die nächsten Zellen laden das Modell und fünf Testfälle. Jeder Testfall enthält eine Frage, einen kurzen Log-Auszug und die richtige Zahl.

Die Funktion `hole_antwort()` liest die Zahl hinter `ANSWER:` aus einer Modellantwort. `bewerte()` vergleicht die Vorhersagen mit den Sollwerten. Beide Funktionen sind bereits fertig; in den Challenges werden nur die Prompts gebaut.

`MAX_ANTWORT_TOKENS = 2400` begrenzt nur die erzeugte Antwort, nicht den Prompt. Das Limit verhindert endlose oder unnötig teure Ausgaben. Es ist hier viermal so groß wie der Helfer-Default, damit auch ein längerer CoT-Pfad noch bis zur Zeile `ANSWER:` kommt.


In [ ]:
# ▶️ Pakete, Pfade und Helferfunktionen
import re
import sys
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                  Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

from helfer import BASIS_URL, MODELL, frage_llm, lade_daten, zeige

print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")


In [ ]:
# ▶️ Testdaten und vorbereitete Bewertung
TESTFAELLE = lade_daten("03_login_testfaelle")[:5]
MAX_ANTWORT_TOKENS = 2400  # Vierfaches des Helfer-Defaults.


def formatiere_aufgabe(fall):
    """Formatiert eine Zählfrage mit ihrem Log-Auszug."""
    return f"Question: {fall['frage']}\n\nLog:\n{fall['log']}"


def hole_antwort(text):
    """Liest die letzte ganze Zahl hinter ANSWER: aus."""
    treffer = re.findall(r"ANSWER\s*:\s*(\d+)", text, flags=re.IGNORECASE)
    return treffer[-1] if treffer else None


def bewerte(prompt_funktion, faelle=TESTFAELLE):
    """Führt einen Prompt auf allen Fällen aus und berechnet die Trefferquote."""
    ergebnisse = []
    for fall in faelle:
        antworttext = frage_llm(
            prompt_funktion(formatiere_aufgabe(fall)), temperature=0.0,
            max_tokens=MAX_ANTWORT_TOKENS
        )
        vorhersage = hole_antwort(antworttext)
        ergebnisse.append({
            "id": fall["id"],
            "soll": fall["antwort"],
            "ist": vorhersage,
            "korrekt": vorhersage == fall["antwort"],
            "antworttext": antworttext,
        })
    quote = sum(e["korrekt"] for e in ergebnisse) / len(ergebnisse)
    return quote, ergebnisse


assert len(TESTFAELLE) == 5
assert hole_antwort("Reasoning ...\nANSWER: 4") == "4"
print(f"{len(TESTFAELLE)} Testfälle geladen. Bewertung ist bereit.")


---
## 1 · Die Aufgabe mit einem normalen Prompt

📖 Unsere einzige Aufgabe lautet: **Zähle die Log-Zeilen, die alle Bedingungen der Frage erfüllen.**

Im Beispiel zählen nur fehlgeschlagene SSH-Logins von `203.0.113.44`. Eine Zeile von einer anderen IP zählt nicht. Ein erfolgreicher Login zählt ebenfalls nicht.

Der normale Prompt verlangt direkt die Endantwort. Er dient als Ausgangspunkt, ist aber keine Challenge.


In [ ]:
# ▶️ Ein anschauliches Beispiel
BEISPIEL = {
    "frage": "How many failed SSH login attempts came from 203.0.113.44?",
    "log": (
        "08:12:03 sshd Failed password from 203.0.113.44\n"
        "08:12:05 sshd Failed password from 198.51.100.9\n"
        "08:12:09 sshd Accepted publickey from 203.0.113.44\n"
        "08:12:14 sshd Failed password from 203.0.113.44"
    ),
}


def normaler_prompt(aufgabe):
    """Fordert nur die Endantwort an."""
    return f"""Count the matching log entries.

{aufgabe}

Return only the result in this exact format:
ANSWER: <integer>"""


aufgabe = formatiere_aufgabe(BEISPIEL)
print(normaler_prompt(aufgabe))


In [ ]:
# ▶️ Baseline einmal ausprobieren
antwort_normal = frage_llm(normaler_prompt(aufgabe), temperature=0.0)
zeige(antwort_normal, titel="Antwort des normalen Prompts")
print("Ausgelesene Zahl:", hole_antwort(antwort_normal))


---
## 2 · Zero-Shot-CoT

📖 Zero-Shot-CoT gibt keine Beispiele vor. Eine kurze Anweisung fordert das Modell auf, die Bedingungen zu identifizieren und den Log schrittweise zu prüfen.

### 🛠️ Challenge 1: Zero-Shot-CoT-Prompt bauen

Implementiere `baue_zero_shot_cot(aufgabe)`.

Der Prompt soll:

- die übergebene Aufgabe enthalten,
- zum schrittweisen Prüfen auffordern,
- mit dem Format `ANSWER: <integer>` enden.

*Tipp: Ergänze den normalen Prompt um eine klare Reasoning-Anweisung.*


In [ ]:
def baue_zero_shot_cot(aufgabe):
    """Baut einen Zero-Shot-CoT-Prompt für eine Log-Zählaufgabe."""
    # TODO: Ergänze Aufgabe, Reasoning-Anweisung und Antwortformat.
    raise NotImplementedError("Challenge 1: Zero-Shot-CoT-Prompt bauen")


In [ ]:
# ✅ Selbsttest
p = baue_zero_shot_cot(aufgabe)
assert isinstance(p, str), "Der Prompt muss ein String sein."
assert aufgabe in p, "Die Aufgabe fehlt im Prompt."
assert "step by step" in p.lower(), "Fordere schrittweises Vorgehen an."
assert "ANSWER: <integer>" in p, "Das feste Antwortformat fehlt."
print("✅ Challenge 1 gelöst")
print(p)


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_zero_shot_cot(aufgabe):
    return f"""Count the matching log entries.

{aufgabe}

Think step by step: identify all conditions, inspect every log line, and keep a running count. Be concise and do not repeat complete log lines.
Finish with the result in this exact format:
ANSWER: <integer>"""
```

</details>


In [ ]:
# ▶️ Zero-Shot-CoT am Einführungsbeispiel
antwort_zero = frage_llm(baue_zero_shot_cot(aufgabe), temperature=0.0)
zeige(antwort_zero, titel="Zero-Shot-CoT")
print("Ausgelesene Zahl:", hole_antwort(antwort_zero))


---
## 3 · Few-Shot-CoT

📖 Few-Shot-CoT zeigt dem Modell gelöste Beispiele mit einem nachvollziehbaren Reasoning-Pfad. Die Beispiele sind bereits gegeben und gehören nicht zu den Testdaten.

Das Muster ist immer gleich:

```text
# Task
<Frage und Log>
# Reasoning
<schrittweise Prüfung>
ANSWER: <Zahl>
```

Danach folgt dieselbe Struktur mit der neuen Aufgabe. Der Bereich `# Reasoning` bleibt offen, damit das Modell ihn fortsetzt.


In [ ]:
# ▶️ Drei vorgegebene Few-Shot-CoT-Beispiele
BEISPIELE = [
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts came from 10.0.0.5?",
            "log": (
                "09:00 status=failed account=root src=10.0.0.5 method=password\n"
                "09:01 status=failed account=root src=10.0.0.9 method=password\n"
                "09:02 status=success account=root src=10.0.0.5 method=password\n"
                "09:03 status=failed account=admin src=10.0.0.5 method=publickey"
            ),
        }),
        "reasoning": (
            "Required: status=failed and src=10.0.0.5. "
            "Line 1 matches, count 1. Line 2 has the wrong source, count 1. "
            "Line 3 has status=success, count 1. Line 4 matches, count 2."
        ),
        "antwort": "2",
    },
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts for admin used publickey?",
            "log": (
                "10:00 status=failed account=admin src=10.0.0.1 method=publickey\n"
                "10:01 status=failed account=root src=10.0.0.2 method=publickey\n"
                "10:02 status=failed account=admin src=10.0.0.3 method=password\n"
                "10:03 status=success account=admin src=10.0.0.4 method=publickey"
            ),
        }),
        "reasoning": (
            "Required: status=failed, account=admin, and method=publickey. "
            "Line 1 matches, count 1. Line 2 has the wrong account, count 1. "
            "Line 3 has the wrong method, count 1. Line 4 has status=success, count 1."
        ),
        "antwort": "1",
    },
    {
        "aufgabe": formatiere_aufgabe({
            "frage": "How many failed login attempts for root came from 192.0.2.4 after 18:00?",
            "log": (
                "17:59 status=failed account=root src=192.0.2.4 method=password\n"
                "18:01 status=failed account=root src=192.0.2.4 method=password\n"
                "18:02 status=failed account=admin src=192.0.2.4 method=password\n"
                "18:03 status=failed account=root src=192.0.2.5 method=password"
            ),
        }),
        "reasoning": (
            "Required: after 18:00, status=failed, account=root, and src=192.0.2.4. "
            "Line 1 is too early, count 0. Line 2 matches, count 1. "
            "Line 3 has the wrong account, count 1. Line 4 has the wrong source, count 1."
        ),
        "antwort": "1",
    },
]

print(BEISPIELE[0]["aufgabe"])
print("\n# Reasoning\n" + BEISPIELE[0]["reasoning"])
print("ANSWER:", BEISPIELE[0]["antwort"])


### 🛠️ Challenge 2: Few-Shot-CoT-Prompt bauen

Implementiere `baue_few_shot_cot(beispiele, aufgabe)`.

Die Funktion soll:

- jedes gelöste Beispiel als `# Task`, `# Reasoning` und `ANSWER:` einfügen,
- danach die neue Aufgabe als `# Task` anhängen,
- offen mit `# Reasoning` enden.

*Tipp: Baue zuerst eine Liste von Beispielblöcken und verbinde sie mit `"\n\n".join(...)`.*


In [ ]:
def baue_few_shot_cot(beispiele, aufgabe):
    """Baut einen Few-Shot-CoT-Prompt aus Beispielen und einer neuen Aufgabe."""
    # TODO: Formatiere Beispiele und offene Aufgabe.
    raise NotImplementedError("Challenge 2: Few-Shot-CoT-Prompt bauen")


In [ ]:
# ✅ Selbsttest
p = baue_few_shot_cot(BEISPIELE, aufgabe)
assert isinstance(p, str), "Der Prompt muss ein String sein."
assert p.count("# Task") == len(BEISPIELE) + 1, "Beispiele plus neue Aufgabe fehlen."
assert p.count("ANSWER:") == len(BEISPIELE) + 1, "Beispiele und Formatanweisung fehlen."
assert "always finish" in p.lower(), "Die Anweisung für das Endformat fehlt."
assert aufgabe in p, "Die neue Aufgabe fehlt."
assert p.rstrip().endswith("# Reasoning"), "Der Prompt muss offen mit # Reasoning enden."
print("✅ Challenge 2 gelöst")
print(p[:900] + "\n...")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def baue_few_shot_cot(beispiele, aufgabe):
    """Baut einen Few-Shot-CoT-Prompt aus Beispielen und einer neuen Aufgabe."""
    anweisung = (
        "Solve the new task by following the exact filtering pattern in the examples. "
        "A field is a filter only when the question explicitly names it; ignore all "
        "other fields. Values such as admin and administrator are different. List every "
        "required condition first, check each line exactly once, keep a running count, "
        "and always finish with ANSWER: <integer>. Be concise."
    )
    bloecke = [
        f"# Task\n{b['aufgabe']}\n\n# Reasoning\n{b['reasoning']}\nANSWER: {b['antwort']}"
        for b in beispiele
    ]
    neue_aufgabe = f"# Task\n{aufgabe}\n\n# Reasoning\n"
    return anweisung + "\n\n" + "\n\n".join(bloecke + [neue_aufgabe])
```

</details>


In [ ]:
# ▶️ Few-Shot-CoT am selben Einführungsbeispiel
antwort_few = frage_llm(lambda_prompt := baue_few_shot_cot(BEISPIELE, aufgabe), temperature=0.0)
zeige(antwort_few, titel="Few-Shot-CoT")
print("Ausgelesene Zahl:", hole_antwort(antwort_few))


---
## 4 · Gegenüberstellung auf den Testdaten

📖 Jetzt laufen alle drei Promptvarianten über dieselben fünf unbekannten Fälle. Die Bewertungsfunktion wurde im Setup vorbereitet. Sie prüft für jeden Fall, ob die Zahl hinter `ANSWER:` exakt dem Sollwert entspricht.

Die normale Variante ist die Baseline. Verglichen werden vor allem die beiden selbst implementierten CoT-Varianten. Da Modellantworten variieren können, sind die konkreten Trefferquoten eine Beobachtung dieses Laufs und keine allgemeine Garantie.


In [ ]:
# ▶️ Derselbe Test für alle Varianten
VARIANTEN = {
    "Normaler Prompt": normaler_prompt,
    "Zero-Shot-CoT": baue_zero_shot_cot,
    "Few-Shot-CoT": lambda aufgabe: baue_few_shot_cot(BEISPIELE, aufgabe),
}

ERGEBNISSE = {}
for name, prompt_funktion in VARIANTEN.items():
    print(f"Teste {name} ...")
    ERGEBNISSE[name] = bewerte(prompt_funktion)

print("\nFertig.")


In [ ]:
# ▶️ Trefferquoten und Einzelergebnisse vergleichen
print(f"{'Variante':<20} {'Trefferquote':>12}")
print("─" * 33)
for name, (quote, _) in ERGEBNISSE.items():
    print(f"{name:<20} {quote:>11.0%}")

print("\nEinzelergebnisse")
print(f"{'Fall':<6} {'Soll':>5} " + " ".join(f"{name[:8]:>9}" for name in VARIANTEN))
print("─" * 46)
for i, fall in enumerate(TESTFAELLE):
    werte = [ERGEBNISSE[name][1][i]["ist"] or "–" for name in VARIANTEN]
    print(f"{fall['id']:<6} {fall['antwort']:>5} " + " ".join(f"{wert:>9}" for wert in werte))

assert all(len(details) == len(TESTFAELLE) for _, details in ERGEBNISSE.values())


📖 **Auswertung:** Prüfe nicht nur die Gesamtquote. Schau in der Tabelle, bei welchen Fällen sich Zero-Shot-CoT und Few-Shot-CoT unterscheiden. Die Reasoning-Texte machen außerdem sichtbar, ob das Modell eine Bedingung übersehen, eine Zeile doppelt gezählt oder `distinct` ignoriert hat.


In [ ]:
# ▶️ Optional: Reasoning eines Testfalls direkt vergleichen
FALL_INDEX = 0
fall = TESTFAELLE[FALL_INDEX]
print(f"Fall {fall['id']} · Sollantwort: {fall['antwort']}")
for name in ["Zero-Shot-CoT", "Few-Shot-CoT"]:
    text = ERGEBNISSE[name][1][FALL_INDEX]["antworttext"]
    zeige(text, titel=name)


---
## 5 · Was du gebaut hast

- Ein normaler Prompt liefert die Baseline für dieselbe Zählaufgabe.
- Zero-Shot-CoT fordert einen schrittweisen Lösungsweg ohne Beispiele an.
- Few-Shot-CoT gibt zwei vollständige Reasoning-Beispiele vor.
- Eine vorbereitete Bewertungsfunktion misst alle Varianten auf denselben Testdaten.

Die zentrale Beobachtung ist nicht, dass CoT immer gewinnt. Der Vergleich zeigt, **ob** und **bei welchen Fällen** ein expliziter Reasoning-Pfad diesem Modell hilft.
